# REINFORCE (Policy Gradient) from Scratch: CartPole-v1

This notebook implements REINFORCE completely from scratch using **PyTorch** and **Gymnasium**.

**What you will build:**
1. CartPole environment recap and stochastic policy intuition
2. Policy Network that outputs action probabilities (not Q-values)
3. Return computation with discounting
4. REINFORCE training loop (episode-level updates)
5. Baseline (normalized returns) for variance reduction
6. Training curves and policy diagnostics
7. Comparison: REINFORCE vs REINFORCE with baseline
8. Ablation: visualizing why variance is the core challenge


## Cell 1 — Imports

In [ ]:
import gymnasium as gym
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torch.distributions import Categorical
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import warnings
warnings.filterwarnings('ignore')

SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device  : {device}')
print(f'PyTorch : {torch.__version__}')
print(f'Gym     : {gym.__version__}')


## Cell 2 — The Fundamental Shift: From Q-Values to Probabilities

Before writing any code, understand the architectural difference.

**DQN output** for CartPole:
```
state [4 floats] → network → [Q(LEFT)=12.3, Q(RIGHT)=14.7]
                              → always pick argmax → RIGHT
```

**REINFORCE output** for CartPole:
```
state [4 floats] → network → [π(LEFT)=0.27, π(RIGHT)=0.73]
                              → SAMPLE from distribution → RIGHT (73% of the time)
```

Key implications:
- The policy is **stochastic** — same state can lead to different actions
- Exploration is **built-in** — no epsilon-greedy needed
- We optimize the policy **directly** via gradient ascent on expected return
- The `Categorical` distribution (PyTorch) handles sampling AND log-probability in one step


In [ ]:
# Demonstrate the categorical distribution — the core building block of REINFORCE
import torch
from torch.distributions import Categorical

# Suppose policy network outputs logits for [LEFT, RIGHT]
logits = torch.tensor([1.2, 2.1])  # raw network outputs (pre-softmax)

# Create categorical distribution
dist = Categorical(logits=logits)

print('=== Categorical Distribution Demo ===')
print(f'Logits          : {logits.numpy()}')
print(f'Probabilities   : {dist.probs.numpy().round(4)}  (sum = {dist.probs.sum():.4f})')
print()

# Sample an action
torch.manual_seed(0)
action = dist.sample()
print(f'Sampled action  : {action.item()} ({"LEFT" if action.item()==0 else "RIGHT"})')
print()

# Compute log probability of the sampled action
log_prob = dist.log_prob(action)
print(f'log π(action)   : {log_prob.item():.4f}')
print(f'  = log({dist.probs[action.item()].item():.4f}) = {torch.log(dist.probs[action.item()]).item():.4f}  ✓')
print()
print('This log_prob is what we scale by the return G_t in the REINFORCE loss.')
print('It is fully differentiable — backprop flows through it to update network weights.')


## Cell 3 — Policy Network

The policy network `π(a|s; θ)` maps state → action probabilities.

Critical differences from the DQN Q-network:
- **Output layer has no explicit activation** (we pass raw logits to `Categorical`)
  - Alternatively, Softmax can be applied — mathematically equivalent
- **Output represents probabilities** (via softmax internally), not values
- **We do NOT take argmax at inference** — we sample (or take argmax for evaluation only)


In [ ]:
class PolicyNetwork(nn.Module):
    """
    Stochastic policy network π(a|s; θ).
    
    Input  : state vector (n_states,)
    Output : action logits (n_actions,)  — pass to Categorical(logits=...)
    
    The network outputs LOGITS (pre-softmax), not probabilities directly.
    PyTorch's Categorical(logits=x) applies log_softmax internally for numerical stability.
    """
    
    def __init__(self, n_states, n_actions, hidden_size=128):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(n_states, hidden_size),
            nn.ReLU(),
            nn.Linear(hidden_size, hidden_size),
            nn.ReLU(),
            nn.Linear(hidden_size, n_actions)
            # No Softmax here — Categorical handles it with better numerical stability
        )
        self._init_weights()
    
    def forward(self, x):
        """Returns logits of shape (..., n_actions)."""
        return self.net(x)
    
    def get_action(self, state):
        """
        Given a single state (numpy array), return:
        - action (int): sampled from π(·|state)
        - log_prob (tensor): log π(action|state) — needed for the loss
        """
        state_t  = torch.tensor(state, dtype=torch.float32).unsqueeze(0).to(device)
        logits   = self.forward(state_t)
        dist     = Categorical(logits=logits)
        action   = dist.sample()
        log_prob = dist.log_prob(action)
        return action.item(), log_prob
    
    def get_action_greedy(self, state):
        """Deterministic greedy action for evaluation (no exploration)."""
        state_t = torch.tensor(state, dtype=torch.float32).unsqueeze(0).to(device)
        with torch.no_grad():
            logits = self.forward(state_t)
        return logits.argmax(dim=-1).item()
    
    def _init_weights(self):
        for layer in self.net:
            if isinstance(layer, nn.Linear):
                nn.init.kaiming_uniform_(layer.weight, nonlinearity='relu')
                nn.init.zeros_(layer.bias)


# ── Instantiate and inspect ──────────────────────────────────────────────────
env = gym.make('CartPole-v1')
n_states  = env.observation_space.shape[0]   # 4
n_actions = env.action_space.n               # 2

policy = PolicyNetwork(n_states, n_actions).to(device)

print('Policy Network:')
print(policy)
print(f'\nParameters: {sum(p.numel() for p in policy.parameters()):,}')
print()

# Forward pass example
state, _ = env.reset(seed=SEED)
action, log_prob = policy.get_action(state)
print(f'State        : {state.round(4)}')
print(f'Logits       : {policy(torch.tensor(state, dtype=torch.float32).unsqueeze(0).to(device)).detach().cpu().numpy().round(4)}')
state_t = torch.tensor(state, dtype=torch.float32).unsqueeze(0).to(device)
probs   = torch.softmax(policy(state_t), dim=-1)
print(f'Probs        : {probs.detach().cpu().numpy().round(4)}  → [P(LEFT), P(RIGHT)]')
print(f'Sampled action: {action} ({"LEFT" if action==0 else "RIGHT"})')
print(f'log π(action): {log_prob.item():.4f}')
env.close()


## Cell 4 — Computing Discounted Returns

After each episode, we compute `G_t` for every timestep `t`:

```
G_T     = 0
G_{T-1} = r_{T-1} + γ · G_T
G_{T-2} = r_{T-2} + γ · G_{T-1}
...
G_0     = r_0 + γ · G_1
```

We iterate **backwards** from the end of the episode. This is a simple cumulative sum with discounting.

**Why discount (γ < 1)?**
- Future rewards are worth less than immediate ones
- Prevents returns from growing unboundedly in long episodes  
- Without discounting (γ = 1), CartPole works fine (episodes end at 500 max), but γ = 0.99 is conventional

**The normalized return baseline:**
After computing all `G_t`, we normalize: `G̃_t = (G_t − mean(G)) / (std(G) + ε)`. This:
- Centers returns around 0: actions in above-average timesteps get positive weight, below-average get negative
- Scales gradients consistently regardless of episode length
- Dramatically reduces training variance


In [ ]:
def compute_returns(rewards, gamma=0.99):
    """
    Compute discounted returns G_t for each step t.
    
    G_t = r_t + γ·r_{t+1} + γ²·r_{t+2} + ... + γ^{T-t}·r_T
    
    Implemented via backward pass for efficiency.
    
    Parameters
    ----------
    rewards : list of floats, one per timestep
    gamma   : discount factor
    
    Returns
    -------
    returns : torch.Tensor of shape (T,), one return per step
    """
    G = 0.0
    returns = []
    for r in reversed(rewards):      # iterate backwards
        G = r + gamma * G
        returns.insert(0, G)         # prepend to maintain forward order
    return torch.tensor(returns, dtype=torch.float32).to(device)


def normalize_returns(returns, eps=1e-8):
    """
    Normalize returns: subtract mean, divide by std.
    Zero-means the advantages — actions above average get +, below get −.
    """
    return (returns - returns.mean()) / (returns.std() + eps)


# ── Visualize what returns look like ─────────────────────────────────────────
# Simulate a short episode where reward = 1 every step
example_rewards = [1.0] * 20   # 20-step episode, all rewards = 1

returns_g99  = compute_returns(example_rewards, gamma=0.99)
returns_g10  = compute_returns(example_rewards, gamma=0.10)
returns_g100 = compute_returns(example_rewards, gamma=1.00)
returns_norm = normalize_returns(returns_g99)

fig, axes = plt.subplots(1, 3, figsize=(14, 4))
fig.suptitle('Discounted Returns for a 20-Step Episode (reward=1 each step)', fontsize=12)

steps = np.arange(20)
axes[0].plot(steps, returns_g100.cpu().numpy(), 'b-o', ms=4, label='γ=1.00 (no discount)')
axes[0].plot(steps, returns_g99.cpu().numpy(),  'g-o', ms=4, label='γ=0.99')
axes[0].plot(steps, returns_g10.cpu().numpy(),  'r-o', ms=4, label='γ=0.10 (heavy discount)')
axes[0].set_xlabel('Timestep t'); axes[0].set_ylabel('G_t')
axes[0].set_title('Effect of Discount Factor γ')
axes[0].legend(fontsize=9); axes[0].grid(True, alpha=0.3)

# Compare short vs long episode
r_short = [1.0] * 10
r_long  = [1.0] * 50
g_short = compute_returns(r_short, 0.99)
g_long  = compute_returns(r_long,  0.99)
axes[1].plot(range(10), g_short.cpu().numpy(), 'b-o', ms=4, label='10-step episode')
axes[1].plot(range(50), g_long.cpu().numpy(),  'g-o', ms=4, label='50-step episode')
axes[1].set_xlabel('Timestep t'); axes[1].set_ylabel('G_t')
axes[1].set_title('Returns Scale With Episode Length')
axes[1].legend(fontsize=9); axes[1].grid(True, alpha=0.3)

# Normalized returns
axes[2].plot(steps, returns_g99.cpu().numpy(),  'b-o', ms=4, label='Raw G_t (γ=0.99)')
axes[2].plot(steps, returns_norm.cpu().numpy(), 'r-o', ms=4, label='Normalized G̃_t')
axes[2].axhline(0, color='gray', linestyle='--', alpha=0.5)
axes[2].set_xlabel('Timestep t'); axes[2].set_ylabel('Return value')
axes[2].set_title('Raw vs Normalized Returns')
axes[2].legend(fontsize=9); axes[2].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print('Key insight from plots:')
print('1. Later timesteps have LOWER G_t (less time left to accumulate reward)')
print('2. Longer episodes have LARGER raw G_t → inconsistent gradient scale')
print('3. Normalized returns center around 0 → consistent scale, no bias')


## Cell 5 — Step-by-Step Trace: One Complete Episode

Before the full training loop, trace through one episode manually to see every tensor and operation.
This is the unit that REINFORCE repeats thousands of times.


In [ ]:
print('=== MANUAL EPISODE TRACE ===')
print()

torch.manual_seed(SEED)
env_trace = gym.make('CartPole-v1')
policy_trace = PolicyNetwork(n_states, n_actions).to(device)
optimizer_trace = optim.Adam(policy_trace.parameters(), lr=1e-3)

state, _ = env_trace.reset(seed=SEED)
log_probs = []   # log π(a_t | s_t) for each step
rewards   = []   # r_t for each step

print(f'{"Step":>5}  {"State (truncated)":>35}  {"Action":>8}  {"logP":>8}  {"Reward":>7}')
print('-' * 70)

for step in range(30):   # run at most 30 steps for illustration
    action, log_prob = policy_trace.get_action(state)
    next_state, reward, terminated, truncated, _ = env_trace.step(action)
    done = terminated or truncated
    
    log_probs.append(log_prob)
    rewards.append(reward)
    
    if step < 8 or done:
        state_str = f'[{state[0]:6.3f} {state[1]:6.3f} {state[2]:6.3f} {state[3]:6.3f}]'
        print(f'{step:>5d}  {state_str:>35}  '
              f'{"LEFT" if action==0 else "RIGHT":>8}  '
              f'{log_prob.item():>8.4f}  {reward:>7.1f}')
    elif step == 8:
        print('  ...')
    
    state = next_state
    if done:
        print()
        print(f'Episode ended at step {step+1}')
        break

# Compute returns
returns = compute_returns(rewards, gamma=0.99)
norm_returns = normalize_returns(returns)

print()
print(f'Episode length     : {len(rewards)} steps')
print(f'Total reward       : {sum(rewards):.0f}')
print()
print(f'Returns G_t (first 5): {returns[:5].cpu().numpy().round(3)}')
print(f'Normalized  (first 5): {norm_returns[:5].cpu().numpy().round(3)}')
print()

# Compute REINFORCE loss
log_probs_t = torch.stack(log_probs)
loss = -(log_probs_t * norm_returns).mean()

print(f'log_probs tensor shape: {log_probs_t.shape}')
print(f'returns tensor shape  : {norm_returns.shape}')
print(f'element-wise product  : shape {(log_probs_t * norm_returns).shape}')
print()
print(f'REINFORCE loss = -mean(log_probs × returns) = {loss.item():.6f}')
print()

# Backprop
optimizer_trace.zero_grad()
loss.backward()

# Show gradient magnitudes for first layer
first_layer_grad = list(policy_trace.net.parameters())[0].grad
print(f'Gradient L2 norm (first layer): {first_layer_grad.norm().item():.6f}')
print()
print('Interpretation:')
print('  - Large |G̃_t| → strong gradient signal at step t')
print('  - Step t where G̃_t > 0 → increase probability of action taken')
print('  - Step t where G̃_t < 0 → decrease probability of action taken')

env_trace.close()


## Cell 6 — REINFORCE Agent Class

Encapsulates the full episode workflow:
1. `collect_episode` — run one full episode, collect log_probs and rewards
2. `compute_loss` — compute REINFORCE loss with optional baseline (normalization)
3. `update` — backpropagate and step optimizer


In [ ]:
class REINFORCEAgent:
    """
    REINFORCE (Monte Carlo Policy Gradient) agent.
    
    Key difference from DQN:
    - No replay buffer (on-policy: must use current policy's trajectories)
    - No target network (no bootstrapping — uses full Monte Carlo returns)
    - Updates once per episode (not per step)
    - Policy is stochastic (samples from distribution)
    """
    
    def __init__(
        self,
        n_states,
        n_actions,
        lr=1e-3,
        gamma=0.99,
        use_baseline=True,    # normalize returns (variance reduction)
        hidden_size=128
    ):
        self.gamma        = gamma
        self.use_baseline = use_baseline
        
        # Policy network — the ONLY network (no target network needed)
        self.policy    = PolicyNetwork(n_states, n_actions, hidden_size).to(device)
        self.optimizer = optim.Adam(self.policy.parameters(), lr=lr)
        
        # Episode storage — cleared after each update
        self.log_probs = []
        self.rewards   = []
        
        # Diagnostics
        self.loss_history           = []
        self.return_history         = []   # G_0 of each episode
        self.grad_norm_history      = []
    
    # ─────────────────────────────────────────────────────────────────────────
    # Act: sample from policy and store log_prob
    # ─────────────────────────────────────────────────────────────────────────
    
    def act(self, state):
        """
        Sample an action from π(·|state) and store log_prob for later update.
        Returns the sampled action (int).
        """
        action, log_prob = self.policy.get_action(state)
        self.log_probs.append(log_prob)
        return action
    
    def store_reward(self, reward):
        """Store reward observed after last action."""
        self.rewards.append(reward)
    
    # ─────────────────────────────────────────────────────────────────────────
    # Update: compute loss and backpropagate (called once per episode)
    # ─────────────────────────────────────────────────────────────────────────
    
    def update(self):
        """
        Core REINFORCE update:
        
        1. Compute discounted returns G_t
        2. Optionally normalize (baseline)
        3. Compute loss = -mean(log_π(a_t|s_t) · G̃_t)
        4. Backpropagate → update policy weights
        5. Clear episode storage
        """
        # ── Step 1: Compute returns ───────────────────────────────────────────
        returns = compute_returns(self.rewards, self.gamma)
        
        # ── Step 2: Optional baseline (normalize returns) ─────────────────────
        if self.use_baseline:
            returns = normalize_returns(returns)
        
        # ── Step 3: Compute REINFORCE loss ────────────────────────────────────
        log_probs_t = torch.stack(self.log_probs)   # (T,)
        
        # Element-wise: log π(a_t|s_t) × G̃_t  for each t
        # Negative because we MAXIMIZE J (gradient ASCENT) but PyTorch
        # minimizes by default → flip sign for gradient DESCENT
        loss = -(log_probs_t * returns).mean()
        
        # ── Step 4: Backpropagate ──────────────────────────────────────────────
        self.optimizer.zero_grad()
        loss.backward()
        
        # Gradient clipping for stability (same as DQN)
        grad_norm = nn.utils.clip_grad_norm_(self.policy.parameters(), max_norm=10.0)
        
        self.optimizer.step()
        
        # ── Diagnostics ───────────────────────────────────────────────────────
        self.loss_history.append(loss.item())
        # G_0: total undiscounted return (sum of raw rewards)
        self.return_history.append(sum(self.rewards))
        self.grad_norm_history.append(grad_norm.item())
        
        # ── Step 5: Clear episode storage ─────────────────────────────────────
        self.log_probs = []
        self.rewards   = []
        
        return loss.item()
    
    def evaluate_greedy(self, env, n_episodes=10):
        """Run greedy evaluation episodes (argmax, no sampling)."""
        rewards = []
        for _ in range(n_episodes):
            s, _ = env.reset()
            ep_r = 0
            for _ in range(500):
                a = self.policy.get_action_greedy(s)
                s, r, t, tr, _ = env.step(a)
                ep_r += r
                if t or tr: break
            rewards.append(ep_r)
        return rewards


# ── Quick sanity check ────────────────────────────────────────────────────────
agent_test = REINFORCEAgent(n_states, n_actions)
print('REINFORCE Agent initialized.')
print(f'  Policy network params: {sum(p.numel() for p in agent_test.policy.parameters()):,}')
print(f'  Optimizer            : Adam (lr=1e-3)')
print(f'  Baseline             : normalize returns (True)')
print()
print('Difference from DQN:')
print('  DQN   → online_net + target_net + replay_buffer + epsilon')
print('  REINFORCE → policy_net only (no separate target, no replay, no epsilon)')


## Cell 7 — Training Loop: REINFORCE with Baseline

The training loop is episodic: each episode completes fully before any weight update.
This is different from DQN which updates every single step.


In [ ]:
# ── Hyperparameters ──────────────────────────────────────────────────────────
GAMMA         = 0.99
LR            = 3e-3
N_EPISODES    = 1000
EVAL_FREQ     = 100   # evaluate every N episodes
EVAL_N        = 10    # greedy episodes per evaluation

# ── Initialize ────────────────────────────────────────────────────────────────
torch.manual_seed(SEED); np.random.seed(SEED)
env_train = gym.make('CartPole-v1')
agent     = REINFORCEAgent(n_states, n_actions, lr=LR, gamma=GAMMA, use_baseline=True)

ep_rewards    = []
ep_lengths    = []
eval_rewards  = []   # mean greedy reward at each eval point
eval_episodes = []

print(f'Training REINFORCE with baseline on CartPole-v1 for {N_EPISODES} episodes...')
print()
print(f'{"Episode":>8}  {"Reward":>8}  {"Steps":>7}  {"Loss":>10}  '
      f'{"GradNorm":>10}  {"AvgR100":>9}')
print('-' * 65)

for episode in range(N_EPISODES):
    state, _ = env_train.reset()
    ep_reward = 0
    
    # ── Collect one full episode ──────────────────────────────────────────────
    for step in range(500):
        action = agent.act(state)                    # sample from π, store log_prob
        state, reward, terminated, truncated, _ = env_train.step(action)
        agent.store_reward(reward)                   # store reward
        ep_reward += reward
        if terminated or truncated:
            break
    
    # ── Update policy (once per episode, using ALL steps) ─────────────────────
    loss = agent.update()
    
    ep_rewards.append(ep_reward)
    ep_lengths.append(step + 1)
    
    # ── Periodic evaluation ───────────────────────────────────────────────────
    if (episode + 1) % EVAL_FREQ == 0:
        eval_r = agent.evaluate_greedy(env_train, n_episodes=EVAL_N)
        eval_rewards.append(np.mean(eval_r))
        eval_episodes.append(episode + 1)
        
        avg_r    = np.mean(ep_rewards[-100:])
        grad_n   = agent.grad_norm_history[-1]
        print(f'{episode+1:>8d}  {ep_reward:>8.0f}  {step+1:>7d}  '
              f'{loss:>10.5f}  {grad_n:>10.4f}  {avg_r:>9.1f}')

print()
print('Training complete!')
print(f'  Final mean reward (last 100 episodes): {np.mean(ep_rewards[-100:]):.1f}')
print(f'  Final greedy eval mean               : {eval_rewards[-1]:.1f}')
env_train.close()


## Cell 8 — Training Curves and Diagnostics

Six diagnostic plots to understand what's happening inside REINFORCE.


In [ ]:
def smooth(data, w=20):
    if len(data) < w: return np.array(data)
    return np.convolve(data, np.ones(w)/w, mode='valid')

fig = plt.figure(figsize=(15, 10))
fig.suptitle('REINFORCE with Baseline — CartPole-v1 Training Diagnostics', fontsize=14, fontweight='bold')
gs  = gridspec.GridSpec(3, 2, figure=fig, hspace=0.4, wspace=0.35)

eps = np.arange(N_EPISODES)

# 1. Episode reward
ax1 = fig.add_subplot(gs[0, 0])
ax1.plot(eps, ep_rewards, alpha=0.2, color='steelblue', linewidth=0.8)
ax1.plot(np.arange(len(smooth(ep_rewards))), smooth(ep_rewards),
         color='steelblue', linewidth=2.5, label='Smoothed (20-ep)')
if eval_episodes:
    ax1.scatter(eval_episodes, eval_rewards, color='red', s=60, zorder=5,
                label='Greedy eval')
ax1.axhline(475, color='red', linestyle='--', alpha=0.5, label='Solved (475)')
ax1.set_title('Episode Reward'); ax1.set_xlabel('Episode'); ax1.set_ylabel('Reward')
ax1.legend(fontsize=8); ax1.grid(True, alpha=0.3)

# 2. Episode length
ax2 = fig.add_subplot(gs[0, 1])
ax2.plot(eps, ep_lengths, alpha=0.2, color='coral', linewidth=0.8)
sm2 = smooth(ep_lengths)
ax2.plot(np.arange(len(sm2)), sm2, color='coral', linewidth=2.5, label='Smoothed (20-ep)')
ax2.axhline(500, color='green', linestyle='--', alpha=0.5, label='Max (500)')
ax2.set_title('Episode Length'); ax2.set_xlabel('Episode'); ax2.set_ylabel('Steps')
ax2.legend(fontsize=8); ax2.grid(True, alpha=0.3)

# 3. Loss
ax3 = fig.add_subplot(gs[1, 0])
losses = agent.loss_history
ax3.plot(eps, losses, alpha=0.2, color='purple', linewidth=0.8)
sm3 = smooth(losses)
ax3.plot(np.arange(len(sm3)), sm3, color='purple', linewidth=2.5)
ax3.set_title('REINFORCE Loss\n(normalized → no clear trend expected)')
ax3.set_xlabel('Episode'); ax3.set_ylabel('Loss')
ax3.grid(True, alpha=0.3)

# 4. Gradient norm
ax4 = fig.add_subplot(gs[1, 1])
gnorms = agent.grad_norm_history
ax4.plot(eps, gnorms, alpha=0.2, color='orange', linewidth=0.8)
sm4 = smooth(gnorms)
ax4.plot(np.arange(len(sm4)), sm4, color='orange', linewidth=2.5)
ax4.set_title('Gradient Norm\n(instability shows as spikes)')
ax4.set_xlabel('Episode'); ax4.set_ylabel('‖∇θ‖')
ax4.grid(True, alpha=0.3)

# 5. Rolling success rate
ax5 = fig.add_subplot(gs[2, 0])
success = [1 if r >= 195 else 0 for r in ep_rewards]
rolling_s = [np.mean(success[max(0,i-50):i+1])*100 for i in range(len(success))]
ax5.plot(eps, rolling_s, color='green', linewidth=2)
ax5.fill_between(eps, rolling_s, alpha=0.1, color='green')
ax5.axhline(90, color='green', linestyle='--', alpha=0.5, label='90%')
ax5.set_title('Rolling Success Rate (reward ≥ 195, 50-ep window)')
ax5.set_xlabel('Episode'); ax5.set_ylabel('Success (%)')
ax5.set_ylim(0, 105); ax5.legend(fontsize=8); ax5.grid(True, alpha=0.3)

# 6. Reward distribution: early vs late
ax6 = fig.add_subplot(gs[2, 1])
half = N_EPISODES // 2
ax6.hist(ep_rewards[:half],  bins=25, alpha=0.6, color='salmon',     label=f'First {half} eps')
ax6.hist(ep_rewards[half:],  bins=25, alpha=0.6, color='steelblue',  label=f'Last {half} eps')
ax6.axvline(475, color='red', linestyle='--', alpha=0.6)
ax6.set_title('Reward Distribution: Early vs Late Training')
ax6.set_xlabel('Reward'); ax6.set_ylabel('Count')
ax6.legend(fontsize=8); ax6.grid(True, alpha=0.3)

plt.show()


## Cell 9 — Policy Internals: What Did the Network Learn?

Visualize the probability the trained policy assigns to each action across the pole angle dimension.

**Expected behavior:** When the pole is tilting right (positive angle), the agent should strongly prefer pushing RIGHT to correct it — and vice versa.


In [ ]:
print('=== TRAINED POLICY INSPECTION ===')
print()

# Test key states
test_cases = [
    ('Upright, still               ', np.array([0.0,  0.0,  0.00,  0.0], dtype=np.float32)),
    ('Tilting right +5°            ', np.array([0.0,  0.0,  0.087, 0.3], dtype=np.float32)),
    ('Tilting right +10°           ', np.array([0.0,  0.0,  0.175, 0.5], dtype=np.float32)),
    ('Tilting left  -5°            ', np.array([0.0,  0.0, -0.087,-0.3], dtype=np.float32)),
    ('Tilting left  -10°           ', np.array([0.0,  0.0, -0.175,-0.5], dtype=np.float32)),
    ('Cart moving right, pole ok   ', np.array([0.5,  1.5,  0.02,  0.1], dtype=np.float32)),
    ('Cart moving left,  pole ok   ', np.array([-0.5,-1.5, -0.02, -0.1], dtype=np.float32)),
]

print(f'{"State description":32}  {"P(LEFT)":>8}  {"P(RIGHT)":>8}  {"Preferred":>10}  {"Confidence":>12}')
print('-' * 80)

for desc, state in test_cases:
    s_t = torch.tensor(state, dtype=torch.float32).unsqueeze(0).to(device)
    with torch.no_grad():
        logits = agent.policy(s_t)
        probs  = torch.softmax(logits, dim=-1).cpu().numpy()[0]
    preferred  = 'LEFT' if probs[0] > probs[1] else 'RIGHT'
    confidence = abs(probs[0] - probs[1])
    print(f'{desc}  {probs[0]:>8.4f}  {probs[1]:>8.4f}  {preferred:>10}  {confidence:>12.4f}')

print()

# ── Sweep pole angle ──────────────────────────────────────────────────────────
angles = np.linspace(-0.35, 0.35, 120)
p_left_vals, p_right_vals = [], []
for angle in angles:
    state = np.array([0.0, 0.0, angle, 0.0], dtype=np.float32)
    s_t   = torch.tensor(state, dtype=torch.float32).unsqueeze(0).to(device)
    with torch.no_grad():
        probs = torch.softmax(agent.policy(s_t), dim=-1).cpu().numpy()[0]
    p_left_vals.append(probs[0])
    p_right_vals.append(probs[1])

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 4))
fig.suptitle('Trained REINFORCE Policy: Action Probabilities', fontsize=12, fontweight='bold')

ax1.plot(np.degrees(angles), p_left_vals,  color='steelblue', linewidth=2.5, label='P(LEFT)')
ax1.plot(np.degrees(angles), p_right_vals, color='coral',     linewidth=2.5, label='P(RIGHT)')
ax1.axvline(0, color='gray', linestyle='--', alpha=0.5, label='Upright')
ax1.axhline(0.5, color='gray', linestyle=':', alpha=0.5)
ax1.set_xlabel('Pole Angle (degrees)'); ax1.set_ylabel('Action Probability')
ax1.set_title('Policy: P(action) vs Pole Angle\n(cart at center, zero velocity)')
ax1.set_ylim(0, 1); ax1.legend(fontsize=9); ax1.grid(True, alpha=0.3)

ax2.plot(np.degrees(angles),
         np.array(p_right_vals) - np.array(p_left_vals),
         color='purple', linewidth=2.5)
ax2.axhline(0, color='gray', linestyle='--', alpha=0.7, label='Indifferent')
ax2.axvline(0, color='gray', linestyle=':', alpha=0.5)
ax2.fill_between(np.degrees(angles),
                 np.array(p_right_vals) - np.array(p_left_vals),
                 0, alpha=0.15, color='purple')
ax2.set_xlabel('Pole Angle (degrees)'); ax2.set_ylabel('P(RIGHT) − P(LEFT)')
ax2.set_title('Action Preference vs Pole Angle\n(positive = prefer RIGHT)')
ax2.legend(fontsize=9); ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()


## Cell 10 — Ablation: REINFORCE with vs Without Baseline

The single most important practical improvement to REINFORCE is return normalization (the baseline).
This cell trains two agents — identical except for the baseline — and compares:
- Learning speed
- Training variance (noisy vs smooth curves)
- Final performance

This directly demonstrates the variance problem described in the tutorial.


In [ ]:
def train_reinforce(use_baseline, n_episodes=800, lr=3e-3, label=''):
    """Train a REINFORCE agent and return reward history."""
    torch.manual_seed(SEED); np.random.seed(SEED)
    env_abl = gym.make('CartPole-v1')
    ag = REINFORCEAgent(n_states, n_actions, lr=lr,
                        gamma=GAMMA, use_baseline=use_baseline)
    rewards = []
    for ep in range(n_episodes):
        state, _ = env_abl.reset()
        ep_reward = 0
        for _ in range(500):
            action = ag.act(state)
            state, reward, t, tr, _ = env_abl.step(action)
            ag.store_reward(reward)
            ep_reward += reward
            if t or tr: break
        ag.update()
        rewards.append(ep_reward)
    env_abl.close()
    final_mean = np.mean(rewards[-100:])
    print(f'  {label:40s} | Final mean (last 100): {final_mean:6.1f}')
    return rewards

print('Training two REINFORCE variants...')
print()
r_with_baseline    = train_reinforce(True,  label='REINFORCE + baseline (normalized returns)')
r_without_baseline = train_reinforce(False, label='REINFORCE (raw returns, no baseline)    ')
print()

# ── Plot comparison ───────────────────────────────────────────────────────────
n_ep = 800
eps  = np.arange(n_ep)

def rolling_mean(data, w=30):
    return [np.mean(data[max(0,i-w):i+1]) for i in range(len(data))]

def rolling_std(data, w=30):
    return [np.std(data[max(0,i-w):i+1]) for i in range(len(data))]

fig, axes = plt.subplots(1, 3, figsize=(15, 5))
fig.suptitle('REINFORCE: Baseline vs No Baseline — Variance Comparison', fontsize=13, fontweight='bold')

# Raw rewards (noisy)
axes[0].plot(eps, r_with_baseline,    alpha=0.15, color='green')
axes[0].plot(eps, r_without_baseline, alpha=0.15, color='red')
axes[0].plot(eps, rolling_mean(r_with_baseline),    color='green', linewidth=2.5, label='With baseline')
axes[0].plot(eps, rolling_mean(r_without_baseline), color='red',   linewidth=2.5, label='No baseline')
axes[0].axhline(475, color='gray', linestyle='--', alpha=0.5)
axes[0].set_title('Rolling Mean Reward (30-ep)')
axes[0].set_xlabel('Episode'); axes[0].set_ylabel('Reward')
axes[0].legend(fontsize=9); axes[0].grid(True, alpha=0.3)

# Rolling variance (key metric!)
rv_with    = rolling_std(r_with_baseline,    30)
rv_without = rolling_std(r_without_baseline, 30)
axes[1].plot(eps, rv_with,    color='green', linewidth=2, label='With baseline')
axes[1].plot(eps, rv_without, color='red',   linewidth=2, label='No baseline')
axes[1].set_title('Rolling Std Dev (30-ep)\n← lower is more stable')
axes[1].set_xlabel('Episode'); axes[1].set_ylabel('Std Dev of Reward')
axes[1].legend(fontsize=9); axes[1].grid(True, alpha=0.3)

# Reward distribution last 200 episodes
axes[2].hist(r_with_baseline[-200:],    bins=20, alpha=0.65, color='green', label='With baseline')
axes[2].hist(r_without_baseline[-200:], bins=20, alpha=0.65, color='red',   label='No baseline')
axes[2].axvline(475, color='gray', linestyle='--', alpha=0.6)
axes[2].set_title('Reward Distribution (Last 200 eps)')
axes[2].set_xlabel('Episode Reward'); axes[2].set_ylabel('Count')
axes[2].legend(fontsize=9); axes[2].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()


## Cell 11 — Visualizing Variance: Why Return Normalization Helps

This cell plots the raw return distributions to build intuition for why high variance is a problem.


In [ ]:
print('=== VARIANCE ANALYSIS: RETURN DISTRIBUTIONS ===')
print()

# Simulate return distributions for episodes of different lengths
# (CartPole: reward=1 per step, so return ≈ episode length)
np.random.seed(SEED)

# Early training: short episodes (mean ~20, std ~15)
early_lengths = np.random.exponential(scale=20, size=500).clip(1, 100).astype(int)
# Late training: long episodes (mean ~450, std ~80)
late_lengths  = np.random.normal(loc=450, scale=80, size=500).clip(50, 500).astype(int)

# Compute G_0 for each episode (approx sum of rewards)
early_G0 = early_lengths.astype(float)
late_G0  = late_lengths.astype(float)

fig, axes = plt.subplots(2, 2, figsize=(13, 9))
fig.suptitle('Understanding Return Variance in REINFORCE', fontsize=13, fontweight='bold')

# Raw return distributions
ax = axes[0, 0]
ax.hist(early_G0, bins=30, alpha=0.7, color='salmon', label='Early training (avg ~20 steps)')
ax.hist(late_G0,  bins=30, alpha=0.7, color='steelblue', label='Late training (avg ~450 steps)')
ax.set_title('Raw Return G_0 Distribution')
ax.set_xlabel('G_0 (total return)'); ax.set_ylabel('Count')
ax.legend(fontsize=9); ax.grid(True, alpha=0.3)
ax.text(0.98, 0.95, f'Early std: {early_G0.std():.1f}\nLate std:  {late_G0.std():.1f}',
        transform=ax.transAxes, ha='right', va='top', fontsize=9,
        bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.5))

# Normalized return distributions
early_norm = (early_G0 - early_G0.mean()) / (early_G0.std() + 1e-8)
late_norm  = (late_G0  - late_G0.mean())  / (late_G0.std()  + 1e-8)
ax = axes[0, 1]
ax.hist(early_norm, bins=30, alpha=0.7, color='salmon',    label='Early (normalized)')
ax.hist(late_norm,  bins=30, alpha=0.7, color='steelblue', label='Late (normalized)')
ax.set_title('Normalized Return G̃_0 Distribution')
ax.set_xlabel('G̃_0'); ax.set_ylabel('Count')
ax.legend(fontsize=9); ax.grid(True, alpha=0.3)
ax.text(0.98, 0.95, f'Early std: {early_norm.std():.3f}\nLate std:  {late_norm.std():.3f}',
        transform=ax.transAxes, ha='right', va='top', fontsize=9,
        bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.5))

# Illustration: gradient step direction over 20 episodes
n_demo = 60
demo_lengths = np.concatenate([early_lengths[:n_demo]])
demo_G0      = demo_lengths.astype(float)
demo_norm    = (demo_G0 - demo_G0.mean()) / (demo_G0.std() + 1e-8)
ax = axes[1, 0]
colors_bar = ['steelblue' if g > 0 else 'coral' for g in demo_norm]
ax.bar(range(n_demo), demo_norm, color=colors_bar, alpha=0.8)
ax.axhline(0, color='black', linewidth=0.8)
ax.set_title('Normalized Returns Over 60 Episodes\n(blue=increase policy prob, red=decrease)')
ax.set_xlabel('Episode'); ax.set_ylabel('Normalized Return')
ax.grid(True, alpha=0.3)

# Effect on effective gradient magnitude
# With large raw returns, gradient magnitude is uncontrolled
raw_grad_magnitudes  = np.abs(demo_G0)
norm_grad_magnitudes = np.abs(demo_norm)
ax = axes[1, 1]
ax.plot(raw_grad_magnitudes,  'coral',     linewidth=1.5, alpha=0.8, label='Raw returns ∝ gradient mag.')
ax.plot(norm_grad_magnitudes, 'steelblue', linewidth=1.5, alpha=0.8, label='Normalized (constant scale)')
ax.set_title('Gradient Magnitude Over 60 Episodes\n(normalized keeps scale controlled)')
ax.set_xlabel('Episode'); ax.set_ylabel('|G_t| (proportional to grad magnitude)')
ax.legend(fontsize=9); ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print('Key takeaways:')
print('1. Raw returns vary from ~1 to ~500 across the training run')
print('2. This means gradient magnitudes swing wildly → inconsistent learning')
print('3. Normalized returns are always ~N(0,1) → consistent, stable updates')
print('4. Normalization also tells the agent which actions were relatively GOOD or BAD')
print('   within an episode, not just whether the episode itself was long or short')


## Cell 12 — Final Evaluation

Run the trained agent with the greedy policy and report performance.


In [ ]:
def evaluate_and_report(agent, n_episodes=30):
    env_eval = gym.make('CartPole-v1')
    rewards  = []
    
    print(f'{"Ep":>4}  {"Reward":>8}  {"Steps":>7}  {"Result":>10}')
    print('-' * 38)
    
    for ep in range(n_episodes):
        state, _ = env_eval.reset()
        ep_reward = 0
        for step in range(500):
            action = agent.policy.get_action_greedy(state)
            state, reward, t, tr, _ = env_eval.step(action)
            ep_reward += reward
            if t or tr: break
        rewards.append(ep_reward)
        result = '✓ Solved' if ep_reward >= 475 else ('~ Good' if ep_reward >= 200 else '✗ Short')
        if ep < 10 or ep_reward < 200:
            print(f'{ep+1:>4d}  {ep_reward:>8.0f}  {step+1:>7d}  {result:>10}')
        elif ep == 10:
            print('  ...')
    
    print()
    print(f'Results over {n_episodes} episodes:')
    print(f'  Mean reward   : {np.mean(rewards):.1f}')
    print(f'  Std dev       : {np.std(rewards):.1f}')
    print(f'  Min / Max     : {np.min(rewards):.0f} / {np.max(rewards):.0f}')
    print(f'  Episodes ≥475 : {sum(r>=475 for r in rewards)}/{n_episodes}')
    print(f'  Episodes =500 : {sum(r==500 for r in rewards)}/{n_episodes}')
    env_eval.close()
    return rewards


print('=== FINAL GREEDY EVALUATION ===')
print()
final_rewards = evaluate_and_report(agent, n_episodes=30)

# Bar chart
fig, ax = plt.subplots(figsize=(12, 4))
colors_eval = ['green' if r >= 475 else ('orange' if r >= 200 else 'red') for r in final_rewards]
ax.bar(range(1, len(final_rewards)+1), final_rewards, color=colors_eval, edgecolor='white', alpha=0.85)
ax.axhline(475, color='red',  linestyle='--', linewidth=1.5, label='Solved (475)')
ax.axhline(np.mean(final_rewards), color='blue', linestyle='-', linewidth=1.5,
           label=f'Mean ({np.mean(final_rewards):.0f})')
legend_patches = [
    mpatches.Patch(color='green',  label='Solved (≥475)'),
    mpatches.Patch(color='orange', label='Good (200–474)'),
    mpatches.Patch(color='red',    label='Short (<200)'),
]
ax.legend(handles=legend_patches, fontsize=9, loc='lower right')
ax.set_xlabel('Evaluation Episode'); ax.set_ylabel('Reward')
ax.set_title('Final Evaluation: Greedy Policy (30 episodes)')
ax.set_ylim(0, 530); ax.set_xticks(range(1, len(final_rewards)+1)); ax.grid(True, alpha=0.3, axis='y')
plt.tight_layout()
plt.show()


## Cell 13 — Three-Way Algorithm Comparison

We have now implemented three RL algorithms on CartPole:
- Q-learning (tabular — not applicable here, but conceptually in the series)
- DQN (value-based, off-policy)
- REINFORCE (policy gradient, on-policy)

This cell runs a fresh REINFORCE and a fresh DQN-style agent (simplified) side by side to compare learning dynamics.


In [ ]:
# Run a quick DQN-style comparison (simplified, no GPU optimization)
class SimpleDQN(nn.Module):
    def __init__(self, n_s, n_a):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(n_s, 128), nn.ReLU(),
            nn.Linear(128, 128), nn.ReLU(),
            nn.Linear(128, n_a)
        )
    def forward(self, x): return self.net(x)

def train_simple_dqn(n_episodes=800):
    import collections
    torch.manual_seed(SEED); np.random.seed(SEED)
    env_d   = gym.make('CartPole-v1')
    net     = SimpleDQN(n_states, n_actions).to(device)
    tgt     = SimpleDQN(n_states, n_actions).to(device)
    tgt.load_state_dict(net.state_dict())
    opt     = optim.Adam(net.parameters(), lr=1e-3)
    buf     = collections.deque(maxlen=10000)
    eps_val = 1.0
    rewards = []
    
    for ep in range(n_episodes):
        s, _ = env_d.reset(); ep_r = 0
        for _ in range(500):
            if np.random.rand() < eps_val:
                a = env_d.action_space.sample()
            else:
                with torch.no_grad():
                    a = net(torch.tensor(s, dtype=torch.float32).unsqueeze(0).to(device)).argmax().item()
            ns, r, t, tr, _ = env_d.step(a)
            done = t or tr
            buf.append((s, a, r, ns, float(done)))
            s = ns; ep_r += r
            if len(buf) >= 64:
                batch = random.sample(buf, 64)
                ss, aa, rr, nss, dd = zip(*batch)
                ss_t  = torch.tensor(np.array(ss),  dtype=torch.float32).to(device)
                aa_t  = torch.tensor(np.array(aa),  dtype=torch.long   ).to(device)
                rr_t  = torch.tensor(np.array(rr),  dtype=torch.float32).to(device)
                nss_t = torch.tensor(np.array(nss), dtype=torch.float32).to(device)
                dd_t  = torch.tensor(np.array(dd),  dtype=torch.float32).to(device)
                with torch.no_grad():
                    mq = tgt(nss_t).max(1).values
                    tg = rr_t + 0.99 * mq * (1 - dd_t)
                pred = net(ss_t).gather(1, aa_t.unsqueeze(1)).squeeze(1)
                loss = F.mse_loss(pred, tg)
                opt.zero_grad(); loss.backward(); opt.step()
            if done: break
        eps_val = max(0.01, eps_val * 0.995)
        if ep % 100 == 0: tgt.load_state_dict(net.state_dict())
        rewards.append(ep_r)
    env_d.close()
    return rewards

print('Training comparison agents (this takes ~2 minutes)...')
r_reinforce = train_reinforce(True,  n_episodes=800, label='REINFORCE w/ baseline')
r_dqn       = train_simple_dqn(800)
print(f'  {"Simple DQN":40s} | Final mean (last 100): {np.mean(r_dqn[-100:]):6.1f}')
print()

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
fig.suptitle('Algorithm Comparison: REINFORCE vs DQN on CartPole-v1', fontsize=13, fontweight='bold')

e800 = np.arange(800)
axes[0].plot(e800, [np.mean(r_reinforce[max(0,i-30):i+1]) for i in range(800)],
             color='steelblue', linewidth=2, label='REINFORCE + baseline')
axes[0].plot(e800, [np.mean(r_dqn[max(0,i-30):i+1]) for i in range(800)],
             color='coral', linewidth=2, label='DQN (simplified)')
axes[0].axhline(475, color='gray', linestyle='--', alpha=0.5, label='Solved (475)')
axes[0].set_title('Rolling Mean Reward (30-ep window)')
axes[0].set_xlabel('Episode'); axes[0].set_ylabel('Mean Reward')
axes[0].legend(fontsize=9); axes[0].grid(True, alpha=0.3)

# Variance comparison (rolling std)
rv_r = [np.std(r_reinforce[max(0,i-30):i+1]) for i in range(800)]
rv_d = [np.std(r_dqn[max(0,i-30):i+1]) for i in range(800)]
axes[1].plot(e800, rv_r, color='steelblue', linewidth=2, alpha=0.8, label='REINFORCE std')
axes[1].plot(e800, rv_d, color='coral',     linewidth=2, alpha=0.8, label='DQN std')
axes[1].set_title('Training Variance (rolling std, 30-ep)\n← REINFORCE is inherently noisier')
axes[1].set_xlabel('Episode'); axes[1].set_ylabel('Std Dev of Reward')
axes[1].legend(fontsize=9); axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print('Observations:')
print('1. DQN typically converges faster (sample efficient via replay buffer)')
print('2. REINFORCE has higher variance even with baseline')
print('3. REINFORCE can match DQN performance but usually needs more episodes')
print('4. REINFORCE naturally handles stochastic policies — useful for exploration and')
print('   continuous action spaces where DQN does not apply')


## Cell 14 — Summary and Next Steps

### What we built from scratch

| Component | Implementation | Key Detail |
|-----------|---------------|------------|
| `PolicyNetwork` | Stochastic π(a\|s;θ) | Outputs logits → `Categorical` for sampling + log_prob |
| `compute_returns` | Monte Carlo G_t | Backward pass, discounted cumsum |
| `normalize_returns` | Baseline / variance reduction | (G - mean) / std — no bias, lower variance |
| `REINFORCEAgent.act` | ε-free exploration | Samples from distribution; stores log_prob |
| `REINFORCEAgent.update` | Policy gradient loss | −mean(log_π × G̃) backprop, once per episode |
| Training loop | Episode-based | Full episode before any update (on-policy constraint) |

### REINFORCE vs DQN: When to Use Which

| Scenario | Prefer REINFORCE | Prefer DQN |
|----------|-----------------|-----------|
| Continuous actions | ✓ (extend with Gaussian policy) | ✗ (Q-values need discrete argmax) |
| Sample efficiency critical | ✗ | ✓ (replay buffer reuses data) |
| Stochastic policy needed | ✓ (naturally stochastic) | ✗ (deterministic by default) |
| Simple discrete tasks | Either | ✓ (faster, more stable) |
| Policy must represent uncertainty | ✓ | ✗ |

### Natural Next Steps

- **Actor-Critic (A2C)**: Replace the mean baseline with a learned value network V(s;φ) — the critic. Uses advantage A_t = G_t − V(s_t) for much lower variance.
- **Proximal Policy Optimization (PPO)**: Adds a clipping constraint to prevent too-large policy updates — the dominant algorithm for continuous control today.
- **SAC (Soft Actor-Critic)**: Off-policy actor-critic with entropy maximization — state-of-the-art for continuous control.
